# Step 0: Environment Setup (Full Databricks Platform)

This notebook sets up the full Databricks environment:
- Installs required Python packages
- Configures Databricks Secrets for API keys
- Sets up Unity Catalog (catalog + schema)
- Verifies Databricks API, Vector Search endpoint, and MLflow

**Run this notebook FIRST before any other notebook.**

## 0A. Install Required Packages

In [0]:
%pip install langchain langchain-community langchain-community langgraph sentence-transformers pydantic mlflow databricks-vectorsearch

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

## 0B. Configure Databricks Target using Databricks Secrets

**First time only -- run these commands once in a separate cell or terminal:**
```
databricks secrets create-scope hackathon
databricks secrets put-secret hackathon DATABRICKS_TOKEN --string-value "your-databricks-token-here"
```

Or use the Databricks UI: Workspace > Secrets > Create Scope > Add Secret

**If secrets are not set up yet, set the key directly below (temporary).**

In [0]:
import os

# Try to get from Databricks Secrets first, fall back to direct setting
try:
    DATABRICKS_TOKEN = dbutils.secrets.get(scope="hackathon", key="DATABRICKS_TOKEN")
    print("Databricks API key loaded from Databricks Secrets")
except Exception:
    # ============================================================
    # FALLBACK: Set your Databricks API key directly here
    # ============================================================
    DATABRICKS_TOKEN = "YOUR_DATABRICKS_TOKEN_HERE"  # <-- Replace this!
    print("WARNING: Using hardcoded API key. Set up Databricks Secrets for production.")

os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN

## 0C. Set Up Unity Catalog

In [0]:
# Create catalog and schema in Unity Catalog
# NOTE: If you don't have permission to create a catalog, use the default catalog
CATALOG_NAME = "hackathon_vf"
SCHEMA_NAME = "healthcare"

try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
    spark.sql(f"USE CATALOG {CATALOG_NAME}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")
    spark.sql(f"USE SCHEMA {SCHEMA_NAME}")
    print(f"Unity Catalog ready: {CATALOG_NAME}.{SCHEMA_NAME}")
except Exception as e:
    # Fallback: use default catalog with hive_metastore
    CATALOG_NAME = "hive_metastore"
    SCHEMA_NAME = "hackathon"
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {SCHEMA_NAME}")
    spark.sql(f"USE {SCHEMA_NAME}")
    print(f"Fallback: using {CATALOG_NAME}.{SCHEMA_NAME}")
    print(f"  (Unity Catalog not available: {e})")

TABLE_PREFIX = f"{CATALOG_NAME}.{SCHEMA_NAME}"
print(f"Table prefix: {TABLE_PREFIX}")

Unity Catalog ready: hackathon_vf.healthcare
Table prefix: hackathon_vf.healthcare


## 0D. Upload Dataset

Upload `dataset.csv` to Databricks:
1. Click "Data" in the left sidebar
2. Click your catalog > schema > "Add Data" or use "Upload to Volume"
3. Or upload to DBFS via "Add Data" > "Upload File"

In [0]:
# Check for dataset in common locations
DATASET_PATH = None
possible_paths = [
    "/FileStore/tables/dataset.csv",
    "/FileStore/dataset.csv",
    "/FileStore/tables/dataset-1.csv",
    "/Workspace/Users/vm8810@srmist.edu.in/dataset.csv"
    f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/data/dataset.csv",
]

for p in possible_paths:
    try:
        dbutils.fs.ls(p)
        DATASET_PATH = p
        print(f"Dataset found at: {p}")
        break
    except Exception:
        continue

if not DATASET_PATH:
    print("Dataset NOT found. Please upload dataset.csv to one of:")
    for p in possible_paths:
        print(f"  {p}")

Dataset NOT found. Please upload dataset.csv to one of:
  /FileStore/tables/dataset.csv
  /FileStore/dataset.csv
  /FileStore/tables/dataset-1.csv
  /Workspace/Users/vm8810@srmist.edu.in/dataset.csv/Volumes/hackathon_vf/healthcare/data/dataset.csv


In [0]:
import pandas as pd

# 1. Read using Pandas (Pandas CAN read workspace files on Serverless)
PANDAS_PATH = "/Workspace/Users/vm8810@srmist.edu.in/dataset.csv"
pdf_raw = pd.read_csv(PANDAS_PATH)

# 2. Convert to Spark DataFrame
df_raw = spark.createDataFrame(pdf_raw)

print(f"Total rows loaded: {df_raw.count()}")


Total rows loaded: 987


## 0E. Verify All Imports

In [0]:
try:
    import pandas as pd
    print("pandas OK")

    from langchain_community.chat_models import ChatDatabricks
    print("langchain-community OK")

    from langchain_core.prompts import ChatPromptTemplate
    print("langchain OK")

    from sentence_transformers import SentenceTransformer
    print("sentence-transformers OK")

    from databricks.vector_search.client import VectorSearchClient
    print("databricks-vectorsearch OK")

    import mlflow
    print("mlflow OK")

    import json, numpy as np
    print("numpy OK")

    print("\nAll imports successful!")
except ImportError as e:
    print(f"ERROR: Missing package - {e}")
    print("Re-run the pip install cell above.")

pandas OK
langchain-community OK
langchain OK
sentence-transformers OK
databricks-vectorsearch OK
mlflow OK
numpy OK

All imports successful!


## 0F. Test Databricks API Connection

In [0]:
from langchain_community.chat_models import ChatDatabricks

llm = ChatDatabricks(
   
    endpoint="databricks-meta-llama-3-3-70b-instruct",
        temperature=0
)

response = llm.invoke("Say 'Databricks connection successful!' and nothing else.")
print(response.content)

Databricks connection successful!


## 0G. Set Up Vector Search Endpoint

Creates a Vector Search endpoint that will be used in notebook 03.

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

VS_ENDPOINT_NAME = "vf_facility_search"

# Check if endpoint exists, create if not
try:
    endpoint = vsc.get_endpoint(VS_ENDPOINT_NAME)
    print(f"Vector Search endpoint '{VS_ENDPOINT_NAME}' already exists. Status: {endpoint.get('endpoint_status', {}).get('state', 'unknown')}")
except Exception:
    try:
        vsc.create_endpoint(name=VS_ENDPOINT_NAME, endpoint_type="STANDARD")
        print(f"Vector Search endpoint '{VS_ENDPOINT_NAME}' created! It may take a few minutes to become ready.")
    except Exception as e:
        print(f"Could not create Vector Search endpoint: {e}")
        print("You may need to create it manually via the Databricks UI: Compute > Vector Search > Create")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Vector Search endpoint 'vf_facility_search' already exists. Status: ONLINE


## 0H. Set Up MLflow Experiment

In [0]:
import mlflow

experiment_name = f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/vf_healthcare_agent"
mlflow.set_experiment(experiment_name)
print(f"MLflow experiment: {experiment_name}")

MLflow experiment: /Users/vm8810@srmist.edu.in/vf_healthcare_agent


In [0]:
print("=" * 60)
print("SETUP COMPLETE!")
print("=" * 60)
print(f"""
Configuration:
  Catalog:         {CATALOG_NAME}
  Schema:          {SCHEMA_NAME}
  Table prefix:    {TABLE_PREFIX}
  Dataset:         {DATASET_PATH}
  VS Endpoint:     {VS_ENDPOINT_NAME}
  LLM:             Databricks (llama-3.1-70b-versatile)
  Databricks Target:    {'Set via Secrets' if 'YOUR' not in DATABRICKS_TOKEN else 'SET MANUALLY (update!)'}

Next: Run notebook 01_data_cleaning
""")

SETUP COMPLETE!

Configuration:
  Catalog:         hackathon_vf
  Schema:          healthcare
  Table prefix:    hackathon_vf.healthcare
  Dataset:         file:/Workspace/Users/vm8810@srmist.edu.in/dataset.csv
  VS Endpoint:     vf_facility_search
  LLM:             Databricks (llama-3.1-70b-versatile)
  Databricks Target:    Set via Secrets

Next: Run notebook 01_data_cleaning

